# Agentic AI: Design → Prototype → Production

**Track:** Applied Agent Engineering Foundations  
**Module:** M4 — Agentic AI  
**Environment:** SageMaker Jupyter Notebook  
**Companion file:** `hr_leave_mcp_server.py`

## What learners will learn
1. Wrap retrieval and actions as tools.
2. Separate knowledge lookup from state-changing actions.
3. Start a small MCP server.
4. Connect an agent to local and MCP-backed tools.
5. Run multi-step workflows.
6. Finish with a mini-hack.

> **Explain to learners:** Agentic systems are not just chatbots. They combine planning, retrieval, action execution, and state handling.


## About the MCP server used in this lab

The provided companion server exposes two leave-related tools:
- `apply_leave`
- `get_leave_summary`

It stores requests in a JSON file and runs with streamable HTTP transport, which makes it a good classroom demo for tool exposure through MCP. fileciteturn3file0


In [ ]:

# Uncomment if your environment needs these libraries
# %pip install -q boto3 pandas numpy python-docx PyPDF2 mcp fastapi uvicorn strands-agents

import json
import os
import subprocess
import time
from io import BytesIO

import boto3
import numpy as np
import PyPDF2
from docx import Document

AWS_REGION = boto3.Session().region_name or "us-east-1"
S3_BUCKET = "YOUR_S3_BUCKET"
S3_PREFIX = "YOUR_DOCUMENT_PREFIX/"   # Example: hr_docs/
EMBED_MODEL_ID = "amazon.titan-embed-text-v1"
TEXT_MODEL_ID = "amazon.nova-lite-v1:0"

HR_MCP_SERVER_PATH = "/home/ec2-user/SageMaker/hr_leave_mcp_server.py"
MCP_SERVER_URL = "http://127.0.0.1:8000/mcp/"

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

print("Region:", AWS_REGION)
print("MCP server path:", HR_MCP_SERVER_PATH)


## Step 1 — Build a lightweight HR RAG tool

For M4, we keep a small local RAG function and combine it with action tools.
This creates a clean separation:

- **Knowledge tool:** answer HR policy questions
- **Action tools:** apply leave, check leave history


In [ ]:

def read_txt_from_s3(bucket: str, key: str) -> str:
    response = s3.get_object(Bucket=bucket, Key=key)
    return response["Body"].read().decode("utf-8", errors="ignore")

def read_docx_from_s3(bucket: str, key: str) -> str:
    response = s3.get_object(Bucket=bucket, Key=key)
    stream = BytesIO(response["Body"].read())
    doc = Document(stream)
    return "\n".join([p.text for p in doc.paragraphs])

def read_pdf_from_s3(bucket: str, key: str) -> str:
    response = s3.get_object(Bucket=bucket, Key=key)
    stream = BytesIO(response["Body"].read())
    reader = PyPDF2.PdfReader(stream)
    return "\n".join([page.extract_text() or "" for page in reader.pages])

def extract_text_from_s3(bucket: str, key: str) -> str:
    if key.lower().endswith(".txt"):
        return read_txt_from_s3(bucket, key)
    if key.lower().endswith(".docx"):
        return read_docx_from_s3(bucket, key)
    if key.lower().endswith(".pdf"):
        return read_pdf_from_s3(bucket, key)
    return ""

def list_document_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.lower().endswith((".txt", ".docx", ".pdf")):
                keys.append(key)
    return keys


In [ ]:

def basic_hr_corpus(bucket: str, prefix: str) -> list[dict]:
    keys = list_document_keys(bucket, prefix)
    docs = []
    for key in keys:
        text = extract_text_from_s3(bucket, key)
        if text.strip():
            docs.append({"source": key, "text": text})
    return docs

def chunk_with_overlap(text: str, chunk_size: int = 600, overlap: int = 120) -> list[str]:
    step = chunk_size - overlap
    return [text[i:i+chunk_size] for i in range(0, len(text), step)]

def get_embedding(text: str) -> list[float]:
    response = bedrock_runtime.invoke_model(
        modelId=EMBED_MODEL_ID,
        body=json.dumps({"inputText": text})
    )
    payload = json.loads(response["body"].read())
    return payload["embedding"]

def cosine_similarity(a, b) -> float:
    a = np.array(a)
    b = np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


In [ ]:

hr_docs = basic_hr_corpus(S3_BUCKET, S3_PREFIX)

hr_chunks = []
for doc in hr_docs:
    for idx, chunk in enumerate(chunk_with_overlap(doc["text"])):
        hr_chunks.append({
            "source": doc["source"],
            "chunk_id": idx,
            "text": chunk
        })

for item in hr_chunks[:min(len(hr_chunks), 30)]:
    item["embedding"] = get_embedding(item["text"])

print("Docs loaded:", len(hr_docs))
print("Chunks embedded for demo:", len([x for x in hr_chunks if "embedding" in x]))


In [ ]:

def hr_rag_search_local(query: str, k: int = 3) -> str:
    query_embedding = get_embedding(query)
    scored = []

    for item in hr_chunks:
        if "embedding" not in item:
            continue
        score = cosine_similarity(query_embedding, item["embedding"])
        scored.append((score, item))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_items = [item for _, item in scored[:k]]

    context = "\n\n".join(
        [f"[Source: {x['source']}]\n{x['text']}" for x in top_items]
    )

    prompt = f'''
You are an enterprise HR assistant.

Answer ONLY from the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}
'''

    response = bedrock_runtime.converse(
        modelId=TEXT_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 250, "temperature": 0.2}
    )
    return response["output"]["message"]["content"][0]["text"]


## Step 2 — Start the MCP server

Make sure the `hr_leave_mcp_server.py` file exists at the expected path.
If needed, copy the uploaded file into your SageMaker working directory first.


In [ ]:

if not os.path.exists(HR_MCP_SERVER_PATH):
    print("MCP server file not found at:", HR_MCP_SERVER_PATH)
    print("Copy the uploaded hr_leave_mcp_server.py file to this location before continuing.")
else:
    print("Server file found.")


In [ ]:

# Start the server in the background only if it is not already running.
# In class, explain why local process management matters for prototyping.

mcp_log_path = "/home/ec2-user/SageMaker/hr_leave_mcp_server.log"

if os.path.exists(HR_MCP_SERVER_PATH):
    process = subprocess.Popen(
        ["python", HR_MCP_SERVER_PATH],
        stdout=open(mcp_log_path, "a"),
        stderr=open(mcp_log_path, "a")
    )
    time.sleep(3)
    print("Started MCP server. PID:", process.pid)
    print("Log file:", mcp_log_path)


## Step 3 — Optional framework-based agent integration

This section uses the same style as your original notebook:
- `Strands Agent`
- local RAG tool
- MCP-provided tools

If the package is unavailable, skip this section and use the fallback planner section later.


In [ ]:

try:
    from strands import Agent
    from strands.tools import tool
    from strands.tools.mcp import MCPClient
    from mcp.client.streamable_http import streamablehttp_client
    STRANDS_AVAILABLE = True
except Exception as e:
    STRANDS_AVAILABLE = False
    print("Strands section unavailable:", e)


In [ ]:

if STRANDS_AVAILABLE:

    @tool
    def hr_rag_search(query: str) -> str:
        """Use this tool for HR policy and HR document questions."""
        return hr_rag_search_local(query)

    mcp_client = MCPClient(lambda: streamablehttp_client(MCP_SERVER_URL))

    agent = Agent(
        model=TEXT_MODEL_ID,
        tools=[hr_rag_search, mcp_client],
        system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR policy or HR document questions.
2. Use MCP leave tools when the user wants to apply leave or asks for leave history.
3. If a request contains both information and action, perform both in sequence.
4. Do not expose internal reasoning.
5. Return only the final answer.
"""
    )

    print("Strands agent ready.")
else:
    print("Skipping Strands agent creation.")


In [ ]:

if STRANDS_AVAILABLE:
    print(agent("What is the leave policy?").message)


In [ ]:

if STRANDS_AVAILABLE:
    print(agent("Apply leave for 2 days for employee EMP001 because of personal work.").message)


In [ ]:

if STRANDS_AVAILABLE:
    print(agent("Check the leave policy and apply leave for 1 day for employee EMP001 due to a family event.").message)


## Step 4 — Add a simple production-style guardrail

We block salary-related questions before running the agent.
This mirrors the guardrail pattern from your original notebook.


In [ ]:

BLOCKED_KEYWORDS = [
    "salary",
    "compensation",
    "payroll",
    "pay band",
    "salary hike",
    "ctc",
    "employee salary"
]

def block_sensitive_queries(query: str) -> None:
    lowered = query.lower()
    if any(k in lowered for k in BLOCKED_KEYWORDS):
        raise ValueError(
            "Policy block: I cannot help with salary, compensation, payroll, or confidential pay information."
        )


In [ ]:

def safe_agent_call(query: str) -> str:
    block_sensitive_queries(query)

    if STRANDS_AVAILABLE:
        response = agent(query)
        return response.message

    # Fallback path if Strands is unavailable:
    # Very lightweight heuristic orchestration for classroom continuity.
    lowered = query.lower()
    if "apply leave" in lowered:
        return "Fallback mode: Strands not available, so MCP action routing is skipped."
    return hr_rag_search_local(query)

print(safe_agent_call("What is the leave policy?"))


In [ ]:

try:
    print(safe_agent_call("Show me employee salary details."))
except Exception as e:
    print("Blocked:", e)


## Step 5 — Add lightweight observability

For classroom use, we capture:
- query
- status
- latency
- final answer


In [ ]:

def observed_run(query: str) -> dict:
    start = time.time()
    try:
        answer = safe_agent_call(query)
        status = "ok"
        error = None
    except Exception as e:
        answer = None
        status = "blocked_or_error"
        error = str(e)

    latency_ms = round((time.time() - start) * 1000, 2)

    return {
        "query": query,
        "status": status,
        "answer": answer,
        "error": error,
        "latency_ms": latency_ms
    }

observed_run("What is the leave policy?")


## Mini-hack — Build a mini HR automation agent

### Minimum requirements
1. Support one policy question.
2. Support one action request.
3. Support one multi-step question that combines policy + action.
4. Block one sensitive query category.

### Good prompts to test
- "What is the leave policy?"
- "Apply leave for 2 days for employee EMP001 because of a family event."
- "Check the leave policy and then apply leave for 1 day."
- "Show me employee salary details."

### Stretch goals
- add persistent memory
- add a second MCP server for ticketing
- log every tool/action call to a CSV


In [ ]:

# --- STUDENT WORK AREA ---
# Suggested extension:
# 1. Add an IT ticket tool or a second MCP server.
# 2. Update the system prompt or fallback planner.
# 3. Demonstrate one combined multi-step workflow.


## Wrap-up

Learners should now understand:
- how local tools and remote tools can coexist
- why MCP is useful for tool interoperability
- how actions differ from knowledge retrieval
- why production patterns include guardrails and observability
